# DrugBank MoA Labeling Pipeline
## Stage 1: Rule-based ActionType mapping → Stage 2: Gemini LLM fallback for ambiguous cases
### Python 3.9 | GPCRactDB v2 compatible

In [ ]:
# Install required packages
# Run this cell only once (skip if already installed)
# !pip install pandas tqdm google-generativeai

In [ ]:
import os
import re
import json
import time
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple

import pandas as pd
from tqdm.notebook import tqdm
import google.generativeai as genai

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.4f}".format)

print("All imports OK")
print(f"pandas version : {pd.__version__}")

In [ ]:
# ============================================================
# USER CONFIG — edit paths and API key here
# ============================================================

# Input file paths
TARGETS_PATH   = "DrugBank_TargetActions_v2.csv"     # cols: DrugBankID, Target_UNIPROT, ActionType
DRUGINFO_PATH  = "DrugBank_Master_Standardized.csv"  # cols: DrugBankID, InChIKey, Name, MechanismOfAction_Text

# Output paths
OUTPUT_PATH      = "DrugBank_MoA_Labeled.csv"
CHECKPOINT_PATH  = "DrugBank_MoA_checkpoint.json"

# Gemini API key — set to None to skip LLM and run rule-based only
GEMINI_API_KEY: Optional[str] = None   # e.g. "AIza..."
# GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# LLM rate-limit base delay (seconds between API calls)
LLM_BASE_DELAY: float = 2.0

# Minimum LLM confidence to trust result; below this threshold → label becomes "binder"
LLM_CONFIDENCE_THRESHOLD: float = 0.5

print("Config loaded.")

In [ ]:
# ============================================================
# Rule-based ActionType → 6-class MoA mapping table
# Coverage: all known DrugBank ActionType string values
# ============================================================

RULE_MAP: Dict[str, str] = {
    # ── Agonist ─────────────────────────────────────────────
    "agonist":                                  "agonist",
    "full agonist":                             "agonist",
    "activator":                                "agonist",
    "stimulator":                               "agonist",
    "inducer":                                  "agonist",
    "opener":                                   "agonist",   # channel opener
    "potentiator":                              "agonist",

    # ── Partial Agonist ──────────────────────────────────────
    "partial agonist":                          "partial agonist",

    # ── Inverse Agonist ──────────────────────────────────────
    "inverse agonist":                          "inverse agonist",

    # ── Antagonist ───────────────────────────────────────────
    "antagonist":                               "antagonist",
    "inhibitor":                                "antagonist",
    "blocker":                                  "antagonist",
    "suppressor":                               "antagonist",
    "inactivator":                              "antagonist",
    "competitive":                              "antagonist",
    "irreversible inhibitor":                   "antagonist",
    "antisense oligonucleotide":                "antagonist",
    "disruptor":                                "antagonist",
    "downregulator":                            "antagonist",
    "chelator":                                 "antagonist",
    "sequestering agent":                       "antagonist",
    "cleavage":                                 "antagonist",

    # ── PAM (Positive Allosteric Modulator) ──────────────────
    "positive allosteric modulator":            "pam",
    "positive modulator":                       "pam",
    "allosteric activator":                     "pam",

    # ── NAM (Negative Allosteric Modulator) ──────────────────
    "negative allosteric modulator":            "nam",
    "negative modulator":                       "nam",
    "allosteric inhibitor":                     "nam",

    # ── Binder (directionally unclear) ───────────────────────
    "binder":                                   "binder",
    "ligand":                                   "binder",
    "substrate":                                "binder",
    "carrier":                                  "binder",
    "cofactor":                                 "binder",
    "chaperone":                                "binder",
    "intercalation":                            "binder",
    "incorporation into and destabilization":   "binder",
    "multitarget":                              "binder",
    "neutralizer":                              "binder",

    # ── Ambiguous → LLM fallback ─────────────────────────────
    "modulator":                                "ambiguous",  # direction unknown → LLM
    "other/unknown":                            "ambiguous",
    "other":                                    "ambiguous",
    "unknown":                                  "ambiguous",
}

# Labels that require LLM re-classification
AMBIGUOUS_LABELS = {"ambiguous", "binder", "modulator"}

# Final 6 functional classes used by GPCRactDB
TARGET_CLASSES = {"agonist", "partial agonist", "inverse agonist", "antagonist", "pam", "nam"}

print(f"RULE_MAP: {len(RULE_MAP)} entries | AMBIGUOUS_LABELS: {AMBIGUOUS_LABELS}")

In [ ]:
def rule_based_label(action_type) -> str:
    """
    Normalize ActionType string and look up in RULE_MAP.
    Returns 'ambiguous' for missing / unrecognized entries,
    which routes those pairs to the LLM stage.
    """
    if pd.isna(action_type) or str(action_type).strip() == "":
        return "ambiguous"
    normalized = str(action_type).lower().strip()
    return RULE_MAP.get(normalized, "ambiguous")


def get_label_source(label: str) -> str:
    """Tag rule-classified labels for provenance tracking."""
    return "ambiguous" if label in AMBIGUOUS_LABELS else "rule"


# Quick sanity check
test_cases = [
    ("agonist",        "agonist"),
    ("Inhibitor",      "antagonist"),
    ("Partial Agonist","partial agonist"),
    ("Modulator",      "ambiguous"),
    (None,             "ambiguous"),
    ("brand new term", "ambiguous"),
]
all_pass = all(rule_based_label(inp) == exp for inp, exp in test_cases)
print(f"Rule-map sanity check: {'PASS' if all_pass else 'FAIL'}")
for inp, exp in test_cases:
    got = rule_based_label(inp)
    status = "OK" if got == exp else f"FAIL (expected {exp})"
    print(f"  {str(inp):<30} -> {got:<20} {status}")

In [ ]:
# ============================================================
# Gemini LLM setup — skip if GEMINI_API_KEY is None
# ============================================================

SYSTEM_PROMPT = """
You are an expert pharmacologist specialized in GPCR drug classification.
Classify the Mechanism of Action (MoA) of a drug acting on a specific target
into exactly ONE of the following 7 labels:

  1. agonist          - Full receptor activator (promotes active-state TM6/7 displacement)
  2. partial agonist  - Activates receptor with submaximal efficacy vs full agonist
  3. inverse agonist  - Suppresses constitutive/basal receptor activity below baseline
  4. antagonist       - Blocks receptor without intrinsic efficacy (competitive or non-competitive)
  5. pam              - Positive Allosteric Modulator: enhances orthosteric agonist response
  6. nam              - Negative Allosteric Modulator: diminishes orthosteric agonist response
  7. binder           - Binding evidence only; functional direction is truly unclear

Classification rules (apply in order):
  - If text describes both activation and inhibition on the SAME receptor -> dominant mechanism wins
  - Do NOT confuse downstream pathway inhibition with receptor-level antagonism
  - For enzyme targets (kinase, protease, etc.) -> treat inhibitor as antagonist
  - 'Modulator' alone without directionality -> binder
  - Absent or uninformative MoA text -> binder

Return ONLY a valid JSON object:
  {"label": "<one of the 7 labels>", "confidence": <0.0-1.0>}
No extra text, no markdown fences.
"""

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    LLM_MODEL = genai.GenerativeModel(
        "models/gemini-2.5-flash-lite",
        system_instruction=SYSTEM_PROMPT,
        generation_config={
            "temperature": 0.0,
            "response_mime_type": "application/json",
        },
    )
    print("Gemini model initialized.")
else:
    LLM_MODEL = None
    print("No API key provided — LLM stage will be skipped (rule-based only).")

In [ ]:
def llm_classify_moa(
    drug_name: str,
    target_uniprot: str,
    action_type_original: str,
    moa_text: str,
    model,
    retries: int = 7,
) -> Dict[str, object]:
    """
    Send a drug-target pair to Gemini for MoA classification.

    Inputs:
        drug_name            - drug common name (e.g. 'Morphine')
        target_uniprot       - UniProt accession (e.g. 'P35372')
        action_type_original - raw DrugBank ActionType string
        moa_text             - DrugBank MechanismOfAction_Text (truncated to 1500 chars)
        model                - initialized GenerativeModel object
        retries              - max retry attempts on rate-limit errors

    Returns:
        dict with keys 'label' (str) and 'confidence' (float)
    """
    user_prompt = (
        f"Drug name: {drug_name}\n"
        f"Target (UniProt): {target_uniprot}\n"
        f"DrugBank ActionType (semi-structured): {action_type_original}\n"
        f"DrugBank MechanismOfAction text:\n\"\"\"{str(moa_text)[:1500]}\"\"\"\n\n"
        "Classify the MoA of this drug on this target."
    )

    wait = 5
    for attempt in range(retries):
        try:
            time.sleep(LLM_BASE_DELAY)
            response = model.generate_content(user_prompt)

            # Strip accidental markdown fences before JSON parsing
            raw = response.text.strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)

            result = json.loads(raw)
            label      = str(result.get("label", "binder")).lower().strip()
            confidence = float(result.get("confidence", 0.5))

            # Apply confidence floor
            if confidence < LLM_CONFIDENCE_THRESHOLD:
                label = "binder"

            return {"label": label, "confidence": confidence}

        except json.JSONDecodeError as e:
            print(f"  [JSON parse error] {e} | raw snippet: {raw[:80]}")
            return {"label": "binder", "confidence": 0.0}

        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "ResourceExhausted" in err_str:
                print(f"  [Rate limit] attempt {attempt + 1}/{retries} — waiting {wait}s")
                time.sleep(wait)
                wait = min(wait * 2, 120)
            else:
                print(f"  [LLM error] {e}")
                break

    return {"label": "binder", "confidence": 0.0}


print("llm_classify_moa() defined.")

In [ ]:
# ============================================================
# Load and merge input files
# ============================================================

print("Loading input files...")
df_targets  = pd.read_csv(TARGETS_PATH)
df_druginfo = pd.read_csv(DRUGINFO_PATH)

print(f"  TargetActions : {len(df_targets):>8,} rows | cols: {df_targets.columns.tolist()}")
print(f"  DrugInfo      : {len(df_druginfo):>8,} rows | cols: {df_druginfo.columns.tolist()}")

# Keep only the columns needed for this pipeline
df_info_slim = (
    df_druginfo[["DrugBankID", "InChIKey", "Name", "MechanismOfAction_Text"]]
    .drop_duplicates(subset="DrugBankID")
)

# Left-join: preserve all target-action rows, attach InChIKey + MoA text
df = df_targets.merge(df_info_slim, on="DrugBankID", how="left")

print(f"\nMerged dataframe : {len(df):,} rows x {df.shape[1]} cols")
print(f"  Rows missing InChIKey      : {df['InChIKey'].isna().sum():,}  (biologics etc.)")
print(f"  Rows missing MoA text      : {df['MechanismOfAction_Text'].isna().sum():,}")
print(f"  Rows missing ActionType    : {df['ActionType'].isna().sum():,}")

df.head(3)

In [ ]:
# ============================================================
# Stage 1 — Rule-based ActionType → 6-class MoA mapping
# ============================================================

df["MoA_Label"]    = df["ActionType"].apply(rule_based_label)
df["Label_Source"] = df["MoA_Label"].apply(get_label_source)
df["LLM_Confidence"] = None  # will be populated in Stage 2

# Distribution overview
print("Stage 1 — Rule-based MoA distribution:")
print("-" * 40)
dist_rule = df["MoA_Label"].value_counts()
for label, cnt in dist_rule.items():
    tag = "✓" if label in TARGET_CLASSES else " "
    print(f"  {tag} {label:<26} {cnt:>8,}")

n_ambiguous = (df["Label_Source"] == "ambiguous").sum()
n_total     = len(df)
print("-" * 40)
print(f"  Functional (6-class) pairs : {(df['MoA_Label'].isin(TARGET_CLASSES)).sum():>8,} / {n_total:,}")
print(f"  Flagged for LLM            : {n_ambiguous:>8,} / {n_total:,}")

In [ ]:
# ============================================================
# Stage 2 — LLM classification for ambiguous pairs
# Skipped automatically if GEMINI_API_KEY is not set
# ============================================================

if LLM_MODEL is None:
    print("LLM stage skipped (no API key).")
else:
    # Load checkpoint (resume support)
    if Path(CHECKPOINT_PATH).exists():
        with open(CHECKPOINT_PATH, "r") as f:
            checkpoint: Dict[str, dict] = json.load(f)
        print(f"Checkpoint loaded: {len(checkpoint):,} pairs already classified.")
    else:
        checkpoint = {}

    ambiguous_idx = df[df["Label_Source"] == "ambiguous"].index
    print(f"Pairs to classify via LLM : {len(ambiguous_idx):,}")

    new_count = 0  # track newly processed entries

    for i, idx in enumerate(tqdm(ambiguous_idx, desc="LLM labeling")):
        row = df.loc[idx]

        # Unique key per drug-target pair for checkpointing
        ck_key = f"{row['DrugBankID']}_{row['Target_UNIPROT']}"

        if ck_key in checkpoint:
            result = checkpoint[ck_key]
        else:
            result = llm_classify_moa(
                drug_name            = str(row.get("Name", "")),
                target_uniprot       = str(row.get("Target_UNIPROT", "")),
                action_type_original = str(row.get("ActionType", "")),
                moa_text             = str(row.get("MechanismOfAction_Text", "")),
                model                = LLM_MODEL,
            )
            checkpoint[ck_key] = result
            new_count += 1

        df.at[idx, "MoA_Label"]      = result["label"]
        df.at[idx, "Label_Source"]   = "llm"
        df.at[idx, "LLM_Confidence"] = result["confidence"]

        # Save checkpoint every 50 new classifications
        if new_count > 0 and new_count % 50 == 0:
            with open(CHECKPOINT_PATH, "w") as f:
                json.dump(checkpoint, f, ensure_ascii=False)
            tqdm.write(f"  Checkpoint saved — {new_count} new entries processed.")

    # Final checkpoint save
    if new_count > 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(checkpoint, f, ensure_ascii=False)
        print(f"\nFinal checkpoint saved: {len(checkpoint):,} total entries.")

    print(f"LLM stage complete. New pairs classified: {new_count:,}")

In [ ]:
# ============================================================
# Post-processing — finalize output columns
# ============================================================

# Flag rows with functional 6-class MoA (usable in GPCRactDB pipeline)
df["Is_Functional"] = df["MoA_Label"].isin(TARGET_CLASSES)

# Final output column selection
OUTPUT_COLS = [
    "DrugBankID", "InChIKey", "Target_UNIPROT",
    "ActionType", "MoA_Label", "Label_Source",
    "LLM_Confidence", "Is_Functional",
]
df_out = df[[c for c in OUTPUT_COLS if c in df.columns]].copy()

# ── Summary stats ────────────────────────────────────────────────────────────
print("=" * 55)
print("Final MoA Distribution")
print("=" * 55)
final_dist = df_out["MoA_Label"].value_counts()
for label, cnt in final_dist.items():
    tag = "✓" if label in TARGET_CLASSES else " "
    print(f"  {tag} {label:<26} {cnt:>8,}")

print()
print("Label Source")
print("-" * 40)
for src, cnt in df_out["Label_Source"].value_counts().items():
    print(f"  {src:<16} {cnt:>8,}")

print()
n_func = df_out["Is_Functional"].sum()
n_total = len(df_out)
print(f"Functional pairs (6-class)     : {n_func:>8,} / {n_total:,}  ({100*n_func/n_total:.1f}%)")

# Unique (InChIKey, UniProt) pairs
df_ikey = df_out.dropna(subset=["InChIKey"])
n_uniq  = df_ikey.drop_duplicates(subset=["InChIKey", "Target_UNIPROT"]).shape[0]
n_func_uniq = (
    df_ikey[df_ikey["Is_Functional"]]
    .drop_duplicates(subset=["InChIKey", "Target_UNIPROT"])
    .shape[0]
)
print(f"Unique (InChIKey, UniProt) pairs: {n_uniq:>8,}")
print(f"  of which functional           : {n_func_uniq:>8,}")
print("=" * 55)

df_out.head(5)

In [ ]:
# ============================================================
# Save final labeled file
# ============================================================

df_out.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}  ({len(df_out):,} rows)")

# Optional: save functional-only subset for direct aggregation
func_path = OUTPUT_PATH.replace(".csv", "_functional_only.csv")
df_out[df_out["Is_Functional"]].to_csv(func_path, index=False)
print(f"Saved: {func_path}  ({df_out['Is_Functional'].sum():,} rows)")

In [ ]:
# ============================================================
# Quick QC — MoA label distribution bar chart
# ============================================================

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Plot 1: all labels ---
    label_counts = df_out["MoA_Label"].value_counts()
    colors = [
        "#2ecc71" if lbl in TARGET_CLASSES else "#bdc3c7"
        for lbl in label_counts.index
    ]
    label_counts.plot(kind="bar", ax=axes[0], color=colors, edgecolor="white")
    axes[0].set_title("All MoA Labels", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("MoA Label")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=40)
    for p in axes[0].patches:
        axes[0].annotate(
            f"{int(p.get_height()):,}",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center", va="bottom", fontsize=8,
        )

    # --- Plot 2: functional 6-class only ---
    func_counts = (
        df_out[df_out["Is_Functional"]]["MoA_Label"]
        .value_counts()
        .reindex(["agonist", "partial agonist", "inverse agonist", "antagonist", "pam", "nam"], fill_value=0)
    )
    class_colors = ["#e74c3c", "#e67e22", "#9b59b6", "#3498db", "#27ae60", "#1abc9c"]
    func_counts.plot(kind="bar", ax=axes[1], color=class_colors, edgecolor="white")
    axes[1].set_title("Functional 6-Class MoA (GPCRactDB)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("MoA Class")
    axes[1].set_ylabel("Count")
    axes[1].tick_params(axis="x", rotation=40)
    for p in axes[1].patches:
        axes[1].annotate(
            f"{int(p.get_height()):,}",
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center", va="bottom", fontsize=8,
        )

    plt.tight_layout()
    plt.savefig("DrugBank_MoA_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved: DrugBank_MoA_distribution.png")

except ImportError:
    print("matplotlib not installed — skipping plot.")